In [1]:
# 1. Install required packages
!pip install -q timm kaggle pyngrok flask flask-cors

import os
import io
import json
import logging
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import timm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from google.colab import userdata, drive
import numpy as np

# Mount Google Drive to access saved weights
drive.mount('/content/drive')

CLASS_NAMES = ['Blotch_Apple', 'Normal_Apple', 'Rot_Apple', 'Scab_Apple']

# Global Configurations (UPDATED)
CONFIG = {
    "img_size": 224,
    "batch_size": 32,
    "epochs": 40,                   # Increased to ensure attention mechanism convergence
    "learning_rate": 3e-5,
    "weight_decay": 0.05,           # Added for L2 regularization
    "model_name": "vit_base_patch16_224",
    "data_dir": "apple_dataset/fruit_disease_dataset/dataset", # Updated to match exact extraction tree
    "save_path": "/content/drive/MyDrive/ViT_Models/real_world_apple_vit_v2.pth"
}

# Set Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Environment ready. Using device: {device}")

Mounted at /content/drive
Environment ready. Using device: cpu


In [2]:
# 2. Image Transformations
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(CONFIG["img_size"], scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

inference_transforms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
print("Transformation pipelines ready.")

Transformation pipelines ready.


In [ ]:
# 3. Setup Kaggle & Download Data
try:
    os.makedirs('/root/.kaggle', exist_ok=True)
    kaggle_config = {
        "username": userdata.get('KAGGLE_USERNAME'),
        "key": userdata.get('KAGGLE_KEY')
    }
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump(kaggle_config, f)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

    print("Downloading dataset from Kaggle...")
    !kaggle datasets download -d anilsandhii/apple-fruit-disease-images-dataset
    print("Extracting...")
    !unzip -q -o apple-fruit-disease-images-dataset.zip -d apple_dataset
    print("Data ready!")
except Exception as e:
    print(f"Kaggle Setup Error: {e}")

Dataset URL: https://www.kaggle.com/datasets/anilsandhii/apple-fruit-disease-images-dataset
License(s): unknown
100% 508M/508M [00:03<00:00, 140MB/s]

Extracting...
Data ready!


In [ ]:
# 4. Train and Save the Vision Transformer (UPDATED)
train_dir = os.path.join(CONFIG["data_dir"], "Train")
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)

# Dynamic Class Weight Calculation
class_counts = [len(os.listdir(os.path.join(train_dir, c))) for c in CLASS_NAMES]
total_samples = sum(class_counts)
class_weights = [total_samples / (len(CLASS_NAMES) * count) for count in class_counts]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Computed Class Weights: {class_weights}")

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2)

train_model = timm.create_model(CONFIG["model_name"], pretrained=True, num_classes=len(CLASS_NAMES))
train_model = train_model.to(device)

# Integrate weights into the loss function
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)

# Apply Weight Decay and Scheduler
optimizer = optim.AdamW(train_model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])

print("Starting Training Loop...")
for epoch in range(CONFIG["epochs"]):
    train_model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = train_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{CONFIG['epochs']} | Loss: {running_loss/len(train_loader):.4f} | LR: {current_lr:.6f}")

# Save the Weights
save_dir = os.path.dirname(CONFIG["save_path"])
if save_dir:
    os.makedirs(save_dir, exist_ok=True)
torch.save(train_model.state_dict(), CONFIG["save_path"])
print(f"Weights successfully saved to: {CONFIG['save_path']}")

Computed Class Weights: [0.8342459983150801, 1.369640387275242, 0.8342459983150801, 1.1461226851851851]
Starting Training Loop...
Epoch 1/40 | Loss: 0.4801 | LR: 0.000030
Epoch 2/40 | Loss: 0.3737 | LR: 0.000030
Epoch 3/40 | Loss: 0.3690 | LR: 0.000030
Epoch 4/40 | Loss: 0.3690 | LR: 0.000029
Epoch 5/40 | Loss: 0.3849 | LR: 0.000029
Epoch 6/40 | Loss: 0.3672 | LR: 0.000028
Epoch 7/40 | Loss: 0.3730 | LR: 0.000028
Epoch 8/40 | Loss: 0.3632 | LR: 0.000027
Epoch 9/40 | Loss: 0.3625 | LR: 0.000026
Epoch 10/40 | Loss: 0.3624 | LR: 0.000026
Epoch 11/40 | Loss: 0.3681 | LR: 0.000025
Epoch 12/40 | Loss: 0.3656 | LR: 0.000024
Epoch 13/40 | Loss: 0.3630 | LR: 0.000023
Epoch 14/40 | Loss: 0.3646 | LR: 0.000022
Epoch 15/40 | Loss: 0.3643 | LR: 0.000021
Epoch 16/40 | Loss: 0.3683 | LR: 0.000020
Epoch 17/40 | Loss: 0.3694 | LR: 0.000019
Epoch 18/40 | Loss: 0.3641 | LR: 0.000017
Epoch 19/40 | Loss: 0.3622 | LR: 0.000016
Epoch 20/40 | Loss: 0.3623 | LR: 0.000015
Epoch 21/40 | Loss: 0.3632 | LR: 0.0000

In [3]:
# 5. Load Model from Storage
print("Building production model architecture...")
live_model = timm.create_model(CONFIG["model_name"], pretrained=False, num_classes=len(CLASS_NAMES))

print(f"Loading weights from {CONFIG['save_path']}...")
live_model.load_state_dict(torch.load(CONFIG["save_path"], map_location=device, weights_only=True))
live_model = live_model.to(device)

# CRITICAL: Put model in evaluation mode for inference
live_model.eval()
print("Live model loaded and ready for production!")

Building production model architecture...
Loading weights from /content/drive/MyDrive/ViT_Models/real_world_apple_vit_v2.pth...
Live model loaded and ready for production!


In [12]:
# 6. Setup Server Endpoint and Tunnel (DUAL CAMERA AGGREGATION)
app = Flask(__name__)
CORS(app)

log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

# --- Global Counters ---
production_stats = {
    "total_processed": 0,
    "Normal_Apple": 0,
    "Blotch_Apple": 0,
    "Rot_Apple": 0,
    "Scab_Apple": 0,
    "Not_Apple_Ignored": 0 # Tracks how many random objects were rejected
}

# --- Confidence Threshold ---
# If the model's confidence is below this, it assumes the object is NOT an apple.
# You may need to tune this number (e.g., 0.60 to 0.85) based on real-world testing.
MIN_CONFIDENCE = 0.50

@app.route('/api/classify', methods=['POST'])
def classify_apple():
    global production_stats

    # 1. Extract both images from the multipart request
    files_to_process = []
    if 'image_csi' in request.files:
        files_to_process.append(("CSI", request.files['image_csi']))
    if 'image_usb' in request.files:
        files_to_process.append(("USB", request.files['image_usb']))

    if len(files_to_process) == 0:
        return jsonify({"error": "No image files provided. Expected 'image_csi' and/or 'image_usb'."}), 400

    results = []

    try:
        # 2. Process all received views
        with torch.no_grad():
            for cam_name, file in files_to_process:
                image = Image.open(io.BytesIO(file.read())).convert('RGB')
                input_tensor = inference_transforms(image).unsqueeze(0).to(device)

                output = live_model(input_tensor)
                probabilities = torch.nn.functional.softmax(output[0], dim=0)
                pred_idx = torch.argmax(probabilities).item()
                confidence = probabilities[pred_idx].item()

                # --- Is it an Apple? Check ---
                if confidence < MIN_CONFIDENCE:
                    assigned_label = "Not_Apple"
                else:
                    assigned_label = CLASS_NAMES[pred_idx]

                results.append({
                    "camera": cam_name,
                    "label": assigned_label,
                    "confidence": confidence
                })

        # 3. Aggregation Logic (Conservative Defect Detection)
        # Separate the views into categories
        not_apple_views = [res for res in results if res['label'] == 'Not_Apple']
        bad_views = [res for res in results if res['label'] not in ['Normal_Apple', 'Not_Apple']]
        good_views = [res for res in results if res['label'] == 'Normal_Apple']

        # --- "Not Apple" Aggregation Logic ---
        # If BOTH cameras agree it's not an apple (or the only camera submitted says it's not an apple)
        if len(not_apple_views) == len(results):
            production_stats["Not_Apple_Ignored"] += 1
            print(f"[SERVER LOG] IGNORING OBJECT. Confidence too low. Likely NOT an apple.")

            return jsonify({
                "class": "ignored",
                "detailed_class": "Not_Apple",
                "confidence": not_apple_views[0]['confidence'],
                "stats": production_stats
            }), 200

        # --- Standard Apple Aggregation Logic ---
        final_quality = "good"
        final_label = "Normal_Apple"
        final_confidence = 0.0

        if len(bad_views) > 0:
            # RULE A: If ANY view is bad, the whole apple is bad.
            final_quality = "bad"
            most_confident_bad = max(bad_views, key=lambda x: x['confidence'])
            final_label = most_confident_bad['label']
            final_confidence = most_confident_bad['confidence']
            print(f"[SERVER LOG] DEFECT DETECTED by {most_confident_bad['camera']} Camera. Apple is BAD ({final_label}).")
        else:
            # RULE B: If ALL views are good, the apple is good.
            final_quality = "good"
            final_label = "Normal_Apple"
            # If one camera said "Not Apple" but the other saw a "Normal Apple", trust the one that saw the apple.
            final_confidence = max([v['confidence'] for v in good_views])
            print(f"[SERVER LOG] ALL CLEAR. Apple is GOOD.")

        # 4. Update the Apple Counters
        production_stats["total_processed"] += 1
        if final_label in production_stats:
            production_stats[final_label] += 1

        print(f"             -> Final Output: {final_label} ({final_confidence*100:.1f}%)")
        print(f"             -> Current Count: Total: {production_stats['total_processed']} | Good: {production_stats['Normal_Apple']} | Blotch: {production_stats['Blotch_Apple']} | Rot: {production_stats['Rot_Apple']} | Scab: {production_stats['Scab_Apple']}\n")

        return jsonify({
            "class": final_quality,
            "detailed_class": final_label,
            "confidence": final_confidence,
            "stats": production_stats
        }), 200

    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Setup Ngrok
try:
    ngrok.kill()
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(5000).public_url
    print(f"\n=========================================")
    print(f" SERVER IS READY. UPDATE FRONTEND C++ TO:")
    print(f" config.backend_url = \"{public_url}/api/classify\";")
    print(f"=========================================\n")
except Exception as e:
    print(f"Ngrok Error: {e}. Check your NGROK_TOKEN in Colab Secrets.")


 SERVER IS READY. UPDATE FRONTEND C++ TO:
 config.backend_url = "https://formidable-olin-unperdurable.ngrok-free.dev/api/classify";



In [13]:
# 7. Start Listening
print("Server is actively listening for Raspberry Pi requests...")
app.run(port=5000)

Server is actively listening for Raspberry Pi requests...
 * Serving Flask app '__main__'
 * Debug mode: off
[SERVER LOG] DEFECT DETECTED by CSI Camera. Apple is BAD (Rot_Apple).
             -> Final Output: Rot_Apple (53.5%)
             -> Current Count: Total: 1 | Good: 0 | Blotch: 0 | Rot: 1 | Scab: 0

[SERVER LOG] DEFECT DETECTED by CSI Camera. Apple is BAD (Blotch_Apple).
             -> Final Output: Blotch_Apple (83.7%)
             -> Current Count: Total: 2 | Good: 0 | Blotch: 1 | Rot: 1 | Scab: 0

[SERVER LOG] ALL CLEAR. Apple is GOOD.
             -> Final Output: Normal_Apple (90.5%)
             -> Current Count: Total: 3 | Good: 1 | Blotch: 1 | Rot: 1 | Scab: 0

